## Sweep analysis

Top-10 runs by selection metric (`best_smoothed_city`) across all 289 evaluated configurations.

In [ ]:
import pandas as pd, numpy as np, wandb

# Load CSV
df = pd.read_csv('sweep_results_final.csv')
df = df.rename(columns={
    'best_smoothed_city':    'best_val_city_rmse',
    'best_val_city_rmse_raw':'city_rmse',
    'base_channels':         'ch',
    'n_layers':              'layers',
    'patch_size':            'patch',
})

# Fetch run names from W&B
PROJECT = 'mjm060323-university-of-amsterdam/cisi-hyperparameter-sweep'

try:
    api  = wandb.Api(timeout=60)
    runs = api.runs(PROJECT, per_page=500)
    name_rows = []
    for r in runs:
        cfg = r.config
        v = r.summary.get('best_smoothed_city')
        if v is None:
            continue
        name_rows.append({
            'run_name':          r.name,
            'best_val_city_rmse': float(v),
            '_lr':               float(cfg.get('lr', 0)),
        })
    names_df = pd.DataFrame(name_rows)
    # merge on rounded selection metric + lr
    df['_key']        = df['best_val_city_rmse'].round(7).astype(str) + '_' + df['lr'].round(9).astype(str)
    names_df['_key']  = names_df['best_val_city_rmse'].round(7).astype(str) + '_' + names_df['_lr'].round(9).astype(str)
    df = df.merge(names_df[['_key', 'run_name']], on='_key', how='left').drop(columns='_key')
    # drop any duplicate columns introduced by merge
    df = df.loc[:, ~df.columns.duplicated()]
    print(f"Run names fetched: {df['run_name'].notna().sum()}/{len(df)}")
except Exception as e:
    print(f"W&B fetch failed ({e}), continuing without run names")
    df['run_name'] = ['run-' + str(i) for i in range(len(df))]

df = df.loc[:, ~df.columns.duplicated()]
print(f"Total runs: {len(df)}  |  columns: {df.columns.tolist()}")


In [ ]:
# Distribution stats
# sel_metric = best_smoothed_city: the EMA-smoothed val city RMSE at the best
# checkpoint epoch, used by W&B Bayesian optimisation to rank and select runs.
# city_rmse = raw (unsmoothed) val city RMSE at that same best epoch.

s = df['best_val_city_rmse']
print(f"N runs : {len(s)}")
print(f"Min    : {s.min():.5f}")
print(f"Max    : {s.max():.5f}")
print(f"Median : {s.median():.5f}")
print(f"Q1     : {s.quantile(0.25):.5f}")
print(f"Q3     : {s.quantile(0.75):.5f}")
print(f"Std    : {s.std():.5f}")
spread_pct = (s.max() - s.min()) / s.min() * 100
print(f"Spread (best to worst): {spread_pct:.1f}%")

# Top-10 table
# sel_metric: smoothed val city RMSE used for run selection
# city_rmse:  raw val city RMSE at the best epoch of that run
top10 = df.nsmallest(10, 'best_val_city_rmse').reset_index(drop=True)
top10.index += 1

print()
print("sel_metric = EMA-smoothed val city RMSE (W&B selection criterion)")
print("city_rmse  = raw val city RMSE at the best checkpoint epoch")
print()
print(f"{'Rank':>4}  {'Run':<22}  {'sel_metric':>10}  {'city_rmse':>9}  {'ch':>4}  {'layers':>6}  {'patch':>5}  {'scale':>5}  {'lr':>12}")
print('-' * 105)
for rank in top10.index:
    r = top10.loc[rank]
    name = str(r['run_name']) if 'run_name' in top10.columns else '?'
    bvr  = float(r['best_val_city_rmse'])
    cr   = float(r['city_rmse'])
    ch   = int(r['ch'])
    lay  = int(r['layers'])
    pat  = int(r['patch'])
    sc   = int(r['scale'])
    lr   = float(r['lr'])
    print(f"{rank:>4}  {name:<22}  {bvr:>10.5f}  {cr:>9.5f}  {ch:>4}  {lay:>6}  {pat:>5}  {sc:>5}  {lr:>12.2e}")

top10_min = float(top10['best_val_city_rmse'].min())
top10_max = float(top10['best_val_city_rmse'].max())
print(f"\nTop-10 band: {top10_min:.5f} – {top10_max:.5f}  (spread {(top10_max-top10_min)/top10_min*100:.1f}%)")
